# 11 · Fill interior gaps, checked against shapely

## What this checks

**Fill interior gaps** removes the holes inside an area's outline, so the area reads as solid. It reports how many
gaps it filled and how much ground that added. This check covers release 0.1.0.

Removing holes is exact. So there is no tolerance to argue about. What the check really tests is the **bookkeeping**:

- the number of gaps filled;
- the area before, with the holes taken out;
- the area after;
- the difference between them.

That difference is ground the travel calculation never found reachable, and the fill now claims. So it is worth
checking.

## How it is checked

We fill the same holes with **shapely**, Python's link to GEOS, the geometry engine much GIS software uses. We project
the areas ourselves with **pyproj**, the Python link to the PROJ projection library. So the areas are recomputed, not
taken on trust. Nothing here uses Fieldwork's code.

The data are the 5-minute and 10-minute corridors from the Broad Street pump, in the John Snow worked example. We
use two bands, not one. The example has a single pump, so one band is one area. With one area, the check could not
tell *filling* holes apart from *joining up* separate areas.

## What should happen

1. **The same number of gaps filled.**
2. **No hole left** in the output.
3. **The same filled shape**, with at most 0.05 m² of ground in one shape but not the other.
4. **The same areas.** The area before, the area after and the area added, each within 0.05 m².

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_11_fill_gaps import run

report = run(Path.cwd() / "fixtures" / "fill-gaps-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["parts"], "parts")
print(f"gaps filled     fieldwork {report['fieldwork']['gapsFilled']}   shapely {report['reference']['gapsFilled']}")
for key, label in [("areaBeforeSquareMetres", "area before"), ("areaAfterSquareMetres", "area after"),
                   ("areaAddedSquareMetres", "area added ")]:
    print(f"{label}     fieldwork {report['fieldwork'][key]:12.2f}   shapely {report['reference'][key]:12.2f}")
print()
print(f"symmetric difference of the filled geometry {report['geometryDifferenceSquareMetres']:.3e} m2")
print(f"interior rings left in the output           {report['interiorRingsRemaining']}")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.1.0, the
current one: **Agrees**.

Both sides filled 60 holes across the two areas, and left none. The filled shapes match exactly: no ground lies in
one but not the other. The areas before, after and added agree to within 0.0083 m². The fill added 32,514 m² to make
an area of 853,069 m².

When first recorded, the corridors had 69 holes and the fill made 938,166 m². The corridors are smaller now, because
Fieldwork charges the walk from the pump onto the street network against the budget. The data changed; the agreement
did not.

Running the cell above repeats the check on the same data.

## What this does not show

- Whether filling is the right thing to do. That question has no numerical answer.
- That the added ground is reachable. The travel calculation never found it so. Our rebuild claims the filled ground
  in the same way, because it is the same operation.